<a href="https://colab.research.google.com/github/s8narnor/Multimodal-AI/blob/main/multimodal_emotion_full_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Multimodal Emotion Recognition

Complete PipelineThis notebook demonstrates an **end‑to‑end multimodal machine learning pipeline** using:

• Audio signals  
• Video facial expressions  

Dataset expected:
Multimodel_Dataset (Audio_Dataset and Video_Dataset)

Pipeline:
1. Load data  
2. Feature extraction  
3. Multimodal feature fusion  
4. Model training  
5. Evaluation  
6. Model saving  
7. Deployment demo

## Install Required Libraries

In [1]:
!pip install librosa opencv-python scipy scikit-learn joblib streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 77.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 101.4 MB/s eta 0:00:00


## Import Libraries

In [15]:
import os
import numpy as np
import librosa
import cv2

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from sklearn.preprocessing import StandardScaler

import joblib

## Dataset Paths

In [3]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("ziya07/multimodal-emotion-recognition-dataset")

print("Path to dataset files:", path)

100%|██████████| 4.73G/4.73G [00:56<00:00, 89.1MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/ziya07/multimodal-emotion-recognition-dataset/versions/2


In [4]:
import shutil
import os

source_directory = "/root/.cache/kagglehub/datasets/ziya07/multimodal-emotion-recognition-dataset/versions/2"
destination_directory = "Multimodel_Dataset"  # Visible directory name (relative path)

# Create the destination directory if it doesn't exist
if not os.path.exists(destination_directory):
    os.makedirs(destination_directory)

# Construct the full destination path
full_destination_path = os.path.join(destination_directory, "1") #Creates a folder called 1 inside the destination folder.

try:
    shutil.copytree(source_directory, full_destination_path)
    print(f"Directory copied to: {full_destination_path}")
except FileExistsError:
    print(f"Destination directory '{full_destination_path}' already exists.")
except FileNotFoundError:
    print(f"Source directory '{source_directory}' not found.")
except Exception as e:
    print(f"An error occurred: {e}")

Directory copied to: Multimodel_Dataset/1


In [16]:
dataset_path = "/content/Multimodel_Dataset/1/archive (18)/Multimodel_Dataset"


audio_path = os.path.join(dataset_path, "Audio_Dataset", "Audio_Dataset")
video_path = os.path.join(dataset_path, "Video_Dataset")

## Audio Feature Extraction (MFCC)

In [17]:
def extract_audio_features(file_path):
    audio, sr = librosa.load(file_path, duration=5)
    mfcc = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=40)
    return np.mean(mfcc.T, axis=0)

## Load Audio Data + Labels

In [18]:
X_audio = []
y = []

label_map = {
    "Normal": 0,
    "Depression/Stage1": 1,
    "Depression/Stage2": 2
}

for label_folder, label in label_map.items():
    folder = os.path.join(audio_path, label_folder)
    for file in os.listdir(folder):
        if file.endswith(".wav"):
            fpath = os.path.join(folder, file)
            features = extract_audio_features(fpath)
            X_audio.append(features)
            y.append(label)

X_audio = np.array(X_audio)
y = np.array(y)

print("Audio samples:", X_audio.shape)

Audio samples: (400, 40)


## Video Feature Extraction

In [21]:
def extract_video_features(video_file):
    cap = cv2.VideoCapture(video_file)
    frame_features = []
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        frame = cv2.resize(frame, (224,224))
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        frame_features.append(np.mean(gray))

    cap.release()
    mean = np.mean(frame_features)
    std = np.std(frame_features)
    return np.array([mean, std])

## Load Video Features

In [22]:
X_video = []

for root, dirs, files in os.walk(video_path):
    for file in files:
        if file.endswith(".mp4"):
            fpath = os.path.join(root, file)
            feature = extract_video_features(fpath)
            X_video.append(feature)

X_video = np.array(X_video)

print("Video samples:", X_video.shape)

Video samples: (240, 2)


## Align Modalities

In [23]:
min_samples = min(len(X_audio), len(X_video))

X_audio = X_audio[:min_samples]
X_video = X_video[:min_samples]
y = y[:min_samples]

print("Aligned dataset size:", min_samples)

Aligned dataset size: 240


## Multimodal Feature Fusion

In [24]:
X = np.concatenate((X_audio, X_video), axis=1)

print("Combined feature shape:", X.shape)

Combined feature shape: (240, 42)


In [25]:
scaler = StandardScaler()

X = scaler.fit_transform(X)

## Train/Test Split

In [26]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

## Train Model

In [27]:
model = RandomForestClassifier(n_estimators=200)
model.fit(X_train, y_train)

RandomForestClassifier(n_estimators=200)

## Model Evaluation

In [28]:
pred = model.predict(X_test)
print(classification_report(y_test, pred))

              precision    recall  f1-score   support

           0       1.00      1.00      1.00        39
           1       1.00      1.00      1.00         9

    accuracy                           1.00        48
   macro avg       1.00      1.00      1.00        48
weighted avg       1.00      1.00      1.00        48



## Save Trained Model

In [31]:
joblib.dump(model, "multimodal_emotion_model.pkl")

['multimodal_emotion_model.pkl']

## Load Model Later

In [32]:
model = joblib.load("multimodal_emotion_model.pkl")

## Simple Prediction Function

In [36]:
emotion_map = {
    0: "Normal",
    1: "Depression Stage 1",
    2: "Depression Stage 2"
}

def predict_emotion(audio_file, video_file):

    a = extract_audio_features(audio_file)
    v = extract_video_features(video_file)

    features = np.concatenate((a, v))
    features = features.reshape(1, -1)

    pred = model.predict(features)[0]

    return emotion_map[pred]

In [37]:
audio_file = '/content/Multimodel_Dataset/1/archive (18)/Multimodel_Dataset/Audio_Dataset/Audio_Dataset/Depression/Stage1/OAF_back_sad.wav'
video_file = '/content/Multimodel_Dataset/1/archive (18)/Multimodel_Dataset/Video_Dataset/Video_Dataset/Video_Speech_Actor_01/Actor_01/01-01-01-01-01-01-01.mp4'

emotion = predict_emotion(audio_file, video_file)

print("Predicted emotion:", emotion)

Predicted emotion: Depression Stage 1


## Deployment (Streamlit)